# BassGen — Python × Pure Data
## A real-time bass line generator based on Markov chains and stochastic rhythm selection

This notebook is the **main entry point** for the BassGen system. It wires together four independent modules and runs the generative loop that communicates with a Pure Data patch over OSC.

| Module | Role |
|---|---|
| `rhythm.py` | **When** notes happen — 16-step velocity-weighted patterns |
| `pitch.py` | **What** notes are allowed — scales, chords, MIDI conversion |
| `markovgen.py` | **How** melodic motion evolves — Markov interval chain |
| `osc_sender/receiver` | **How** events are emitted and controls are received |

### Execution order
1. **Imports & config** — run once per session
2. **Server restart** (optional) — only needed if the OSC server is already running
3. **OSC server** — starts the listener thread
4. **Main loop** — blocking; interrupt with the kernel stop button
5. **Test cells** — independent, can be run in any order without Pure Data

## 1 · Imports & configuration

In [ ]:
from rhythm import RHYTHM_PATTERNS, choose_rhythm_pattern, pattern_to_attacks
from pitch import NOTE_TO_PC, SCALE_PATTERNS, CHORD_TYPES, build_scale, get_chord_midi_notes
from markovgen import MARKOV_INTERVAL_TRANSITIONS, initial_bass_note, next_bass_note
from osc_sender import OSCSender
import osc_receiver

import time
import threading
from pythonosc.osc_server import ThreadingOSCUDPServer

In [ ]:
# ---------------------------------------------------------------------------
# Network configuration
# Python listens for OSC control messages from Pure Data on PYTHON_LISTEN_PORT.
# Python sends note events to Pure Data on port 8000 (set inside OSCSender).
# ---------------------------------------------------------------------------
PYTHON_LISTEN_IP   = "127.0.0.1"
PYTHON_LISTEN_PORT = 9000


def step_time_from_bpm(bpm: float) -> float:
    """Return the duration of one 16th-note step in seconds for a given BPM."""
    return (60.0 / bpm) / 4.0

## 2 · OSC server

The server runs in a **daemon thread** so it shuts down automatically when the kernel stops.  
If you need to restart it mid-session (e.g. after changing `osc_receiver.py`), run the *Server restart* cell first, then this one.

In [ ]:
# ---------------------------------------------------------------------------
# Server restart — run this only if the server is already running and you
# need to reset it (e.g. after editing osc_receiver.py).
# Skip on first run; 'server' won't exist yet.
# ---------------------------------------------------------------------------
try:
    server.shutdown()
    server.server_close()
    print("Previous server stopped.")
except NameError:
    print("No server running — skipping shutdown.")

In [ ]:
# Start the OSC listener in a background daemon thread.
# Incoming messages update module-level state in osc_receiver.py,
# which the main loop reads on every tick.
server = ThreadingOSCUDPServer(
    (PYTHON_LISTEN_IP, PYTHON_LISTEN_PORT),
    osc_receiver.dispatcher
)

osc_thread = threading.Thread(target=server.serve_forever, daemon=True)
osc_thread.start()

print(f"OSC server listening on {PYTHON_LISTEN_IP}:{PYTHON_LISTEN_PORT}")

## 3 · Main generative loop

This cell blocks until the kernel is interrupted.  
Pure Data must be running and sending a `/transport/start` message before notes will be produced.

**Loop anatomy (one tick = one 16th-note step):**
1. Check `bass_enabled`; skip the tick silently if muted.
2. At step 0 (bar boundary): apply any pending key / degree / rhythm changes.
3. Read the current step's velocity weight from the active pattern.
4. If weight > 0: calculate velocity, send the note via OSC, then advance the Markov state.
5. Sleep until the next step.

In [ ]:
# ---------------------------------------------------------------------------
# Initial musical state — edit these to change the starting key/mode/chord.
# All parameters can also be changed live from Pure Data via OSC.
# ---------------------------------------------------------------------------
key            = 'D'
key_type       = 'harmonic_minor'   # major | dorian | mixolydian | natural_minor | harmonic_minor
chord_type     = 'triad'            # triad | seventh
current_degree = 0                  # 0-based scale degree (0 = tonic)

# Build the initial harmonic context
scale_pcs  = build_scale(key, key_type)
chord_midi = get_chord_midi_notes(key, key_type, current_degree, chord_type)
chord_pcs  = [n % 12 for n in chord_midi]

print(f"Key: {key} {key_type}  |  scale PCs: {scale_pcs}")
print(f"Degree {current_degree} chord MIDI notes: {chord_midi}")

# Initialise Markov state
current_note      = initial_bass_note(chord_midi)  # start on the lowest chord tone
previous_interval = 0                              # treat the first step as 'came from rest'

# Default rhythm pattern (overridden at bar boundaries by OSC)
pattern = RHYTHM_PATTERNS['habanera']

osc = OSCSender()

# ---------------------------------------------------------------------------
# Main loop — one iteration per 16th-note step.
# Interrupt the kernel (■) to stop.
# ---------------------------------------------------------------------------
while True:

    if not osc_receiver.bass_enabled:
        # Generator is muted from PD — spin without sleeping to stay responsive.
        continue

    step_time = step_time_from_bpm(osc_receiver.bpm)

    # -- Bar boundary: apply pending parameter changes ----------------------
    if osc_receiver.step == 0:

        if osc_receiver.pending_degree is not None:
            current_degree = osc_receiver.pending_degree - 1  # PD sends 1-based
            osc_receiver.pending_degree = None

        if osc_receiver.pending_rhythm is not None:
            pattern = RHYTHM_PATTERNS[osc_receiver.pending_rhythm]
            osc_receiver.pending_rhythm = None

        if osc_receiver.pending_key is not None:
            key = osc_receiver.pending_key
            osc_receiver.pending_key = None

        if osc_receiver.pending_keyType is not None:
            key_type = osc_receiver.pending_keyType
            osc_receiver.pending_keyType = None

        # Recompute harmonic context with the (possibly updated) parameters
        chord_midi = get_chord_midi_notes(key, key_type, current_degree, chord_type)
        chord_pcs  = [n % 12 for n in chord_midi]

    # -- Per-step note generation -------------------------------------------
    step = osc_receiver.step
    hit  = pattern[step]          # velocity weight for this step (0 = rest)

    if hit > 0:
        # Steps 0 and 8 are downbeats — restrict to chord tones only.
        # All other steps allow scale tones as well.
        beat_strength = 1.0 if step in (0, 8) else 0.5

        duration = step_time - 0.0005          # slight staccato to avoid note overlap
        velocity = max(1, min(127, int(hit * 127)))

        osc.send_note(current_note, duration, velocity, step)

        # Advance Markov state for the *next* step
        current_note, previous_interval = next_bass_note(
            prev_note      = current_note,
            prev_interval  = previous_interval,
            chord_pcs      = chord_pcs,
            scale_pcs      = scale_pcs,
            beat_strength  = beat_strength
        )

    # Advance step counter and sleep until the next 16th note
    osc_receiver.step = (step + 1) % 16
    time.sleep(step_time - 0.0005)

---
## 4 · Tests

The cells below are **independent** of the main loop and Pure Data.  
Run them individually to inspect or debug each subsystem.

### 4a · Rhythm engine
Randomly selects four patterns and prints their step indices (attack positions).

In [ ]:
for bar in range(4):
    pat_name = choose_rhythm_pattern()
    pattern  = RHYTHM_PATTERNS[pat_name]
    attacks  = pattern_to_attacks(pattern)

    print(f"Bar {bar + 1}: pattern = '{pat_name}'")
    print(f"  Raw weights : {pattern}")
    print(f"  Attack steps: {attacks}")
    print()

### 4b · Harmonic context
Builds a chord from a key + scale + degree and prints the resulting MIDI notes.

In [ ]:
# Arguments: root note, scale type, degree (0-based), chord quality
test_key    = "C"
test_scale  = "major"
test_degree = 1        # degree II → D minor in C major
test_chord  = "triad"

notes = get_chord_midi_notes(test_key, test_scale, test_degree, test_chord)
pcs   = [n % 12 for n in notes]

print(f"{test_key} {test_scale}, degree {test_degree} ({test_chord})")
print(f"  MIDI notes      : {notes}")
print(f"  Pitch classes   : {pcs}")

### 4c · OSC sender (no PD feedback)
Runs the full generator loop and sends notes to PD, but ignores incoming OSC messages.  
Useful for checking that the OSC → PD connection works with a fixed rhythm and key.  
⚠️ Requires `SECONDS_PER_16TH` to be set and PD to be running.

In [ ]:
BPM             = 120
SECONDS_PER_16TH = (60 / BPM) / 4

osc = OSCSender()

scale_pcs  = build_scale("C", "major")
chord_midi = get_chord_midi_notes("C", "major", 4, "triad")  # degree IV = F major
chord_pcs  = [n % 12 for n in chord_midi]

current_note      = initial_bass_note(chord_midi)
previous_interval = 0

while True:
    pattern = RHYTHM_PATTERNS["habanera"]

    for step, hit in enumerate(pattern):
        if hit > 0:
            beat_strength = 1.0 if step in (0, 8) else 0.5
            velocity      = max(1, min(127, int(hit * 127)))

            osc.send_note(current_note, SECONDS_PER_16TH, velocity, step)

            current_note, previous_interval = next_bass_note(
                prev_note     = current_note,
                prev_interval = previous_interval,
                chord_pcs     = chord_pcs,
                scale_pcs     = scale_pcs,
                beat_strength = beat_strength
            )

        time.sleep(SECONDS_PER_16TH)

### 4d · OSC receiver
Prints the current `bass_enabled` state every 250 ms.  
Send `/bass/enable 1` from PD and confirm the message is received.

In [ ]:
while True:
    status = "ACTIVE" if osc_receiver.bass_enabled else "muted"
    print(f"bass_enabled = {osc_receiver.bass_enabled}  [{status}]")
    time.sleep(0.25)

### 4e · Direct MIDI output (no Pure Data required)
Plays the generator through `pygame.midi` directly — useful for auditioning melodic output without a PD patch.  
Requires `pygame` (`pip install pygame`) and a MIDI output device or software synth on port 0.

In [ ]:
import pygame.midi


class MidiOut:
    """Minimal pygame.midi wrapper for blocking note playback.

    Args:
        instrument: General MIDI program number (default: 33 = Acoustic Bass).
    """

    def __init__(self, instrument: int = 33):
        pygame.midi.init()
        self.port = pygame.midi.Output(0)
        self.port.set_instrument(instrument)

    def note_on(self, note: int, velocity: int = 90) -> None:
        self.port.note_on(note, velocity)

    def note_off(self, note: int) -> None:
        self.port.note_off(note)

    def play_note(self, note: int, duration: float, velocity: int = 90) -> None:
        """Send note-on, block for *duration* seconds, then send note-off."""
        self.note_on(note, velocity)
        time.sleep(duration)
        self.note_off(note)

    def close(self) -> None:
        self.port.close()
        pygame.midi.quit()

In [ ]:
BPM              = 120
SECONDS_PER_16TH = (60 / BPM) / 4

midi = MidiOut()  # instrument 33 = Acoustic Bass

scale_pcs  = build_scale("C", "major")
chord_midi = get_chord_midi_notes("C", "major", 3, "seventh")  # degree IV = F dominant 7
chord_pcs  = [n % 12 for n in chord_midi]

current_note  = initial_bass_note(chord_midi)
prev_interval = 0

try:
    while True:
        pattern = RHYTHM_PATTERNS["habanera"]

        for step, hit in enumerate(pattern):
            if hit:
                beat_strength = 1.0 if step in (0, 8) else 0.5
                duration      = SECONDS_PER_16TH * 1.5  # slightly legato
                velocity      = 90

                midi.play_note(current_note, duration, velocity)

                current_note, prev_interval = next_bass_note(
                    current_note, prev_interval, chord_pcs, scale_pcs, beat_strength
                )

            time.sleep(SECONDS_PER_16TH)

except KeyboardInterrupt:
    print("Stopped. Closing MIDI port.")
    midi.close()